# 11 · Model identification: BE or SC?

**What this shows.** Which of the two candidate strategies better explains each animal's trial-by-trial
updates — boundary estimation (BE: a boundary learnt from reward) or statistical categorisation (SC: a learnt
stimulus distribution). Both models are fitted to the expert sessions of notebook 10 and scored on held-out
data, with two independent methods (grid search, amortised SBI) and a consensus. On the real cohort first
(all mice); a synthetic cohort, where the truth is known, is the check that the methods recover it.

This is a description of the animals, not a factor in the opto analyses: with 9 opto animals and few SC calls,
splitting the opto effects by strategy would be anecdotal.

**How a call is made.** The manuscript's model selection, copied: the cross-validation is repeated over seeds,
and the model with the lower held-out error is called when a paired Wilcoxon over the repeats gives p < 0.05
(the manuscript used an ANOVA over 64 seeds). The repeats vary the simulation noise and the posterior samples,
not the data split, so a significant call means the call is stable under that noise.

**What produced it.** One `model_identification` run per cohort, on the cluster (see `START_HERE.md`,
"Model identification", and `docs/runs.md`); fetch it with `sc-fetch results model_identification/all-mice`.
The run holds `grid_search/<fit_target>/<distribution>/`, `sbi/...` and `consensus/<distribution>/assignments.csv`.

**Whether the methods can be trusted** is notebook 91 (appendix): calibration and coverage of the SBI networks,
parameter recovery, model recovery on a synthetic cohort of 50 BE + 50 SC animals shaped like the real ones and
run with this run's seeds and repeats, the check that the networks match the current code, and the two
consensus rules side by side. Notebook 90 is where the SBI summary statistics were chosen.


In [ ]:
from nb_setup import *
from sound_categorisation.inference.cv_utils import load_cv_results
from sound_categorisation.plotting.cv import plot_confusion, plot_cv_comparison, plot_winner_summary
from sound_categorisation.data.paths import model_id_dir
banner()
run = open_run('model_identification', MODEL_COHORT)
DIST, FT = 'uniform', 'update_matrix'
gs = load_cv_results(model_id_dir(run, 'grid_search', FT, DIST))
print(f'grid search: {gs.comparison.animal_id.nunique()} animals, {gs.long.seed.nunique()} seeds')

## 1 · Grid search: held-out error of each model per animal

For every animal both models are fitted by grid search with block-aware folds; the score is the held-out
MSE of the predicted update matrix. `winner` is the lower mean; `p_value` the paired Wilcoxon over seeds
(stability under simulation noise, as above).

In [ ]:
gs.comparison.round(4)

In [ ]:
aid = gs.comparison.animal_id.iloc[0]
plot_cv_comparison(gs.long, gs.comparison, aid, fit_target='UM');

## 2 · Recovery on the synthetic cohort

If `true_model` is in the comparison table (synthetic cohorts), the confusion matrix is the recovery
rate of this method at this cohort size. With the `--fast` settings CI uses this is a pipe-clean, not a
power estimate.
The thesis numbers (every voter and the consensus, both rules, with intervals) are in notebook 91.


In [ ]:
if 'true_model' in gs.comparison and gs.comparison.true_model.notna().any():
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
    plot_confusion(gs.comparison, ax=axes[0])
    axes[1].remove(); axes[1] = fig.add_subplot(1, 2, 2)
    w = plot_winner_summary(gs.comparison)
    plt.close(w)
    acc = gs.comparison.correct.mean()
    axes[1].bar(['correct', 'wrong'], [acc, 1 - acc], color=['0.4', '0.8']); axes[1].set(ylim=(0, 1), title=f'GS recovery {acc:.2f}')
    plt.tight_layout()
else:
    print('real cohort: no ground truth; see the consensus below')

## 3 · SBI, when present

The amortised networks (`sc-sbi-train`) are conditioned on each animal's expert data (`sc-sbi-condition`),
and the posterior-predictive held-out error is scored in the same schema, so `load_cv_results` reads it
unchanged. Absent in CI (no torch) — the cell says so rather than failing.

In [ ]:
sbi_dirs = sorted((run / 'sbi' / FT / DIST).glob('*')) if (run / 'sbi' / FT / DIST).exists() else []
if sbi_dirs:
    for d in sbi_dirs:
        r = load_cv_results(d)
        print(f'SBI rep {d.name}:'); display(r.comparison.round(4))
else:
    print('no SBI results in this run (sc-sbi-condition needs torch + trained networks)')

## 4 · Consensus

`sc-consensus` collects each method's significant call (p < 0.05) and applies the rule. The primary rule calls the
model with more significant votes, needing at least one; a tie is Split, no significant vote is Unclear. A call
made with a significant vote against it (2 vs 1) stands but is flagged `contested`; `n_sig_BE` / `n_sig_SC` are the
counts behind each call. The stricter rule (two significant votes) is shown below for comparison.

In [ ]:
asg, summary = load_consensus(run, DIST)
print(summary)
cols = [c for c in asg.columns if not c.endswith('_p')]
asg[cols]

In [ ]:
from sound_categorisation.inference.consensus import load_all_assignments
two = load_all_assignments(run, DIST, min_significant_votes=2)
side = asg[['id', 'Consensus', 'contested', 'n_sig_BE', 'n_sig_SC']].merge(
    two[['id', 'Consensus']].rename(columns={'Consensus': 'two votes'}), on='id')
print('one significant vote:', side['Consensus'].value_counts().to_dict())
print('two significant votes:', side['two votes'].value_counts().to_dict())
print('contested calls:', ', '.join(side.loc[side['contested'], 'id']) or 'none')
side

## 5 · Absolute fit: each model against the noise floor

The floor is the error between the update matrices of the two halves the fits are scored on: sessions 1–3 against
4–5 for the grid search and the pooled network, the first half of each session's trials against the second for the
single-session network. Both halves are the same animal, so the floor is the noise of a few hundred trials plus the
drift between days. A fit is scored on one simulation of itself, as noisy as the held-out half, so a correct model
scores about the floor: each model's held-out error is shown over it. About 1: as good as a model with fixed
parameters can be on these data. Well above 1: the model misses something. The no-history reference is the
animal's own psychometric curve with no dependence on the previous trial, scored the same way: it carries the
average of the animal's updating, not its trial-to-trial dependence, so its expected update matrix is zero.

The grey band is where a correct model's error fell (10–90 %) on the synthetic validation cohort's floor run, when
there is one (`sc-noise-floor --cohort <validation cohort>`). From `sc-noise-floor --cohort <cohort>`
(`noise_floor/<cohort>/latest`).

In [ ]:
from sound_categorisation.inference.noise_floor import floor_ratios, model_errors, true_model_band
from sound_categorisation.plotting.noise_floor import plot_noise_floor

nf = pd.read_csv(open_run('noise_floor', MODEL_COHORT) / 'noise_floor.csv')
try:
    ref = pd.read_csv(open_run('noise_floor', VALIDATION_COHORT) / 'noise_floor.csv')
except FileNotFoundError:
    ref = nf
methods = {'grid search': ('session', gs.comparison)}
for d in sbi_dirs:
    methods[f'SBI {d.name}'] = ('within' if d.name == 'single' else 'session', load_cv_results(d).comparison)
fig, axes = plt.subplots(len(methods), 1, figsize=(10, 3.4 * len(methods)), squeeze=False)
ratios = {}
for ax, (name, (split, comp)) in zip(axes[:, 0], methods.items()):
    ratios[name] = floor_ratios(nf, model_errors(comp), split)
    plot_noise_floor(ratios[name], band=true_model_band(ref, split), ax=ax)
    ax.set_title(f'{name}: {split} split', fontsize=9)
plt.tight_layout()
pd.concat({k: v.set_index('animal') for k, v in ratios.items()}, axis=1).round(2)

## 6 · Predicting the Hard sessions from the Uniform fits

Uniform data alone may not tell BE from SC (§5). Here the fitted models are run on the Hard sessions and compared
with what each mouse did there. `sc-hard-predictions` takes a mouse's fitted Uniform sessions and goes straight into
its Hard sessions; what lies between is left out (`sessions.csv` lists it). For each of 200 draws of a model's SBI
posterior, a copy of the mouse plays that span in one continuous run, with the mouse's own stimuli, learning from
feedback and carrying what it learnt from one day into the next. The mouse and every copy go through the switch
report's own computation, so data and models are measured the same way; the models have no overnight change.

Readouts: for the blocked design (behaviour1), per switch the PSE before it, the block's PSE, the block's share of the
normative shift and the accuracy on hard stimuli, and per mouse the mean block PSE of Hard-A minus Hard-B; for the
daily alternation (opto1), per mouse and kind of day the convergence over trials 100–300, with the model learning on
laser trials or not. Black: the mouse; colour: each model's median over the draws, bar 10–90 %. The table sums the
CRPS (lower is better) per model over the mice, the mice the bias check flags apart. Five clean blocked mice give
per-mouse results, not a group test.

In [ ]:
from sound_categorisation.plotting.hard_predictions import plot_readout, plot_trajectory

hard = {}
for cohort in dict.fromkeys([SWITCH_COHORT, OPTO_COHORT]):
    try:
        hard[cohort] = open_run('hard_predictions', cohort)
    except FileNotFoundError as e:
        print(e)
if hard:
    HS = pd.concat([pd.read_csv(r / 'scores.csv') for r in hard.values()], ignore_index=True)
    HS['kind'] = HS['kind'].fillna('')
    HT = pd.concat([pd.read_csv(r / 'totals.csv') for r in hard.values()], ignore_index=True)
    HT['kind'] = HT['kind'].fillna('')
    wanted = [('pse_block', None), ('block_convergence', None), ('hard_a_minus_hard_b', None),
              ('convergence_100_300', 'masking day'), ('convergence_100_300', 'laser day')]
    panels = [(r, k) for r, k in wanted if ((HS['readout'] == r) & ((HS['kind'] == k) if k else True)).any()]
    if panels:
        tallest = max(len(HS.loc[HS['readout'] == r, ['animal', 'switch_idx']].drop_duplicates()) for r, _ in panels)
        fig, axes = plt.subplots(1, len(panels), figsize=(4.6 * len(panels), 0.32 * tallest + 2), squeeze=False)
        for ax, (r, k) in zip(axes[0], panels):
            plot_readout(HS, r, kind=k, ax=ax)
        plt.tight_layout()
    display(HT.pivot_table(index=['design', 'readout', 'kind', 'group'], columns=['model', 'laser_learning'],
                           values='crps_sum').round(3))

In [ ]:
# the criterion within the first switches of one blocked and one daily mouse; plot_trajectory(TR, animal, switch)
# draws any other
if hard:
    TR = pd.concat([pd.read_csv(r / 'trajectories.csv') for r in hard.values()], ignore_index=True)
    shown = []
    for design in ('blocked', 'daily'):
        d = TR[(TR['design'] == design) & (TR['source'] == 'observed')]
        if len(d):
            aid = d['animal'].iloc[0]
            shown += [(aid, int(k)) for k in sorted(d.loc[d['animal'] == aid, 'switch_idx'].unique())[:3]]
    if shown:
        fig, axes = plt.subplots(1, len(shown), figsize=(3.6 * len(shown), 3), squeeze=False, sharey=True)
        for ax, (aid, k) in zip(axes[0], shown):
            plot_trajectory(TR, aid, k, ax=ax)
        plt.tight_layout()

## 7 · Reading the real cohort

With `SC_NB_SYNTHETIC` off this notebook shows the all-mice run: §1, §4, §5 and §6 are the real animals and
§2 is skipped (no ground truth). An Unclear call means neither model was reliably better for that animal, not that
the animal uses neither strategy.